# Camada Bronze

In [0]:
# Defina o catálogo e o schema de trabalho
nome_catalogo = 'senai'
nome_schema = 'medallion'

# Cria o catálogo e o schema caso ele ainda não exista
spark.sql(f'CREATE CATALOG IF NOT EXISTS {nome_catalogo}')
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {nome_catalogo}.{nome_schema}')

print(f'Schema criado: {nome_catalogo}.{nome_schema}')

Schema criado: senai.medallion


In [0]:
# Leitura batch da tabela de origem
origem_dados = 'samples.nyctaxi.trips'
df = spark.table(origem_dados)

display(df.limit(10))
df.printSchema()

tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,pickup_zip,dropoff_zip
2016-02-13T21:47:53.000Z,2016-02-13T21:57:15.000Z,1.4,8.0,10103,10110
2016-02-13T18:29:09.000Z,2016-02-13T18:37:23.000Z,1.31,7.5,10023,10023
2016-02-06T19:40:58.000Z,2016-02-06T19:52:32.000Z,1.8,9.5,10001,10018
2016-02-12T19:06:43.000Z,2016-02-12T19:20:54.000Z,2.3,11.5,10044,10111
2016-02-23T10:27:56.000Z,2016-02-23T10:58:33.000Z,2.6,18.5,10199,10022
2016-02-13T00:41:43.000Z,2016-02-13T00:46:52.000Z,1.4,6.5,10023,10069
2016-02-18T23:49:53.000Z,2016-02-19T00:12:53.000Z,10.4,31.0,11371,10003
2016-02-18T20:21:45.000Z,2016-02-18T20:38:23.000Z,10.15,28.5,11371,11201
2016-02-03T10:47:50.000Z,2016-02-03T11:07:06.000Z,3.27,15.0,10014,10023
2016-02-19T01:26:39.000Z,2016-02-19T01:40:01.000Z,4.42,15.0,10003,11222


root
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- pickup_zip: integer (nullable = true)
 |-- dropoff_zip: integer (nullable = true)



In [0]:
# Gravação da camada Bronze
tabela_bronze = 'senai.medallion.bronze_trips'

(
    df.write
        .mode('overwrite')
        .saveAsTable(tabela_bronze)
)

print(f'Total de registros: {spark.table(tabela_bronze).count()}')

Total de registros: 21932


### Atividade prática

Agora faça a ingestão da camada Bronze com Spark.

Tarefa:
- ler a tabela `samples.bakehouse.sales_transactions`
- criar a tabela `senai.medallion.senai.bronze_sales`
- ao final, mostrar a quantidade de registros carregados

In [0]:
# Fonte da atividade
origem_dados_ativ = 'samples.bakehouse.sales_transactions'

# Tabela a ser criada
tabela_bronze_ativ = 'senai.medallion.bronze_sales'

In [0]:
# Atividade Camada Bronze
df_ativ1 = spark.table(origem_dados_ativ)

df_ativ1.write.mode('overwrite').saveAsTable(tabela_bronze_ativ)

total_registros = spark.table(tabela_bronze_ativ).count()

print(f'Total de registros: {total_registros}')

Total de registros: 3333


# Camada Silver

In [0]:
# Tabelas da utilizadas
tabela_bronze = 'senai.medallion.bronze_trips'
tabela_silver = 'senai.medallion.silver_trips'

# Lê a Bronze
df_bronze = spark.table(tabela_bronze)
display(df_bronze.limit(10))

tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,pickup_zip,dropoff_zip
2016-02-13T21:47:53.000Z,2016-02-13T21:57:15.000Z,1.4,8.0,10103,10110
2016-02-13T18:29:09.000Z,2016-02-13T18:37:23.000Z,1.31,7.5,10023,10023
2016-02-06T19:40:58.000Z,2016-02-06T19:52:32.000Z,1.8,9.5,10001,10018
2016-02-12T19:06:43.000Z,2016-02-12T19:20:54.000Z,2.3,11.5,10044,10111
2016-02-23T10:27:56.000Z,2016-02-23T10:58:33.000Z,2.6,18.5,10199,10022
2016-02-13T00:41:43.000Z,2016-02-13T00:46:52.000Z,1.4,6.5,10023,10069
2016-02-18T23:49:53.000Z,2016-02-19T00:12:53.000Z,10.4,31.0,11371,10003
2016-02-18T20:21:45.000Z,2016-02-18T20:38:23.000Z,10.15,28.5,11371,11201
2016-02-03T10:47:50.000Z,2016-02-03T11:07:06.000Z,3.27,15.0,10014,10023
2016-02-19T01:26:39.000Z,2016-02-19T01:40:01.000Z,4.42,15.0,10003,11222


In [0]:
# Limpeza inicial da tabela:
# - remover linhas com valores nulos
# - remover linhas duplicadas
df_limpa = df_bronze.dropna().dropDuplicates()

In [0]:
from pyspark.sql.functions import to_date, date_format, unix_timestamp, round, col

# Transformação da tabela silver:
# - criar a data da corrida
df_silver = (
    df_limpa
    .withColumn('pickup_date', to_date(col("tpep_pickup_datetime")))

# - criar a hora de início
    .withColumn('pickup_hour', date_format(col('tpep_pickup_datetime'), 'HH:mm'))

# - calcular a duração em minutos
    .withColumn(
        'trip_duration_minutes',
        round(
            (unix_timestamp(col('tpep_dropoff_datetime')) - unix_timestamp(col('tpep_pickup_datetime'))) / 60,
            2
        )
    )
)

# Gravação da Silver
df_silver.write.mode('overwrite').saveAsTable(tabela_silver)

In [0]:
# Visualização das novas colunas da tabela Silver
display(
    spark.table(tabela_silver)
        .select('pickup_date', 'pickup_hour', 'trip_duration_minutes')
        .limit(10)
)

pickup_date,pickup_hour,trip_duration_minutes
2016-02-18,07:32,4.97
2016-02-18,10:14,7.63
2016-02-15,01:45,3.48
2016-02-20,19:28,40.35
2016-02-20,09:12,9.23
2016-02-12,22:11,5.7
2016-02-07,00:15,10.3
2016-02-17,13:57,7.03
2016-02-04,09:21,10.53
2016-02-26,07:16,7.73


### Atividade prática

Agora prepare os dados da Bronze de vendas e execute algumas tansformações para  gravar a tabela Silver.

Tarefa:
- ler a tabela `senai.medallion.bronze_sales`
- remover transações duplicadas com base em `transactionID`
- utilize o método `.filter()` para manter apenas registros com `quantity`, `unitPrice` e `totalPrice` maiores que zero
- criar a coluna `date` com a data da transação
- utilize o método `when()` criar a coluna `pagCartao` indicando se existe número de cartão

In [0]:
# Fonte da atividade
tabela_bronze_ativ = 'senai.medallion.bronze_sales'

# Tabela a ser criada
tabela_silver_ativ = 'senai.medallion.silver_sales'

In [0]:
from pyspark.sql.functions import col, to_date, hour, month, when

# Atividade da Camada Silver
df_ativ2 = spark.table(tabela_bronze_ativ)

# Limpeza dos dados
df_limpa_ativ = (
    df_ativ2
        .dropDuplicates(['transactionID'])
        .filter(col('quantity') > 0)
        .filter(col('unitPrice') > 0)
        .filter(col('totalPrice') > 0)
)

# Colunas com atributos derivadas
df_silver_ativ = (
    df_limpa_ativ
        .withColumn('date', to_date(col('dateTime')))
        .withColumn('pagCartao',
                    when(col('cardNumber').isNotNull(), 1).otherwise(0)
    )
)

In [0]:
# Gravação da tabela Silver
(
    df_silver_ativ.write
        .mode('overwrite')
        .saveAsTable(tabela_silver_ativ)
)

# Visualização da tabela
display(spark.table(tabela_silver_ativ).select('date', 'pagCartao').limit(10))

date,pagCartao
2024-05-02,1
2024-05-09,1
2024-05-04,1
2024-05-12,1
2024-05-06,1
2024-05-01,1
2024-05-07,1
2024-05-14,1
2024-05-11,1
2024-05-04,1


# Camada Gold

In [0]:
%sql
-- TODO

### Atividade prática

Agora crie uma tabela Gold a partir de `senai.medallion.silver_sales`.

Tarefa:
- criar a tabela `senai.medallion.gold_sales`
- agrupar por `franchiseID`
- calcular:
  - quantidade de transações
  - quantidade total de itens vendidos
  - valor total vendido
  - ticket médio
- ao final, consultar a tabela criada

In [0]:
%sql
-- Atividade da Camada Gold
